# FinChain 03 - Rejection SFT and OPD

This notebook tests whether the model benefits more from verified positives alone or from chosen/rejected pairs sampled from its own current behavior.

OPD is the bridge from DPO to RLVR: DPO-style loss, but on-policy verifier-labeled pairs.


In [ ]:
from pathlib import Path
import importlib
import json
import os
import platform
import subprocess
import sys
import time

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = Path("/workspace/finpost") if Path("/workspace/finpost").exists() else PROJECT_ROOT

RESULTS_DIR = PROJECT_ROOT / "results" / "finchain_rlvr"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def progress(title, detail=None):
    stamp = time.strftime("%H:%M:%S")
    print(f"[{stamp}] {title}")
    if detail:
        print(detail)

def check_module(name):
    try:
        return importlib.import_module(name), None
    except Exception as exc:
        return None, exc

def run_cmd(cmd, *, check=False):
    progress("running command", cmd)
    completed = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if check and completed.returncode != 0:
        raise RuntimeError(f"command failed with exit {completed.returncode}: {cmd}")
    return completed

def append_cost_event(stage, **payload):
    path = RESULTS_DIR / "cost_ledger.jsonl"
    row = {"stage": stage, "time": time.strftime("%Y-%m-%dT%H:%M:%S"), **payload}
    with path.open("a", encoding="utf-8") as fp:
        fp.write(json.dumps(row, sort_keys=True) + "\n")
    print(json.dumps(row, indent=2, sort_keys=True))
    return row

progress("project root", str(PROJECT_ROOT))
progress("python", sys.version.split()[0])
progress("platform", platform.platform())


In [ ]:
progress("GPU preflight")
try:
    import torch
    print("torch:", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        for idx in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(idx)
            print(f"gpu {idx}: {props.name}, vram={props.total_memory / 1e9:.1f} GB")
except Exception as exc:
    print("torch check failed:", repr(exc))

run_cmd("nvidia-smi")


In [ ]:
progress("repo dependency preflight")
for module_name in [
    "finpost.training.trainer",
    "finpost.training.dpo",
    "finpost.training.preference_data",
]:
    module, exc = check_module(module_name)
    print(module_name, "OK" if module else f"MISSING: {exc}")

for planned in [
    "finpost.data.finchain",
    "finpost.evals.finchain_metrics",
    "finpost.posttraining.rollout_cache",
    "finpost.posttraining.verifier",
    "finpost.posttraining.opd",
    "finpost.posttraining.grpo",
]:
    module, exc = check_module(planned)
    print(planned, "OK" if module else "planned, not implemented yet")


In [ ]:
progress("OPD pair construction sketch")
rollouts = [
    {"prompt_id": "p1", "completion": "correct chain", "verified": True, "bucket": "ambiguous"},
    {"prompt_id": "p1", "completion": "wrong chain", "verified": False, "bucket": "ambiguous"},
]
chosen = [r for r in rollouts if r["verified"]]
rejected = [r for r in rollouts if not r["verified"]]
print("chosen", len(chosen), "rejected", len(rejected), "pairs", min(len(chosen), len(rejected)))


In [ ]:
METHOD_TABLE = [
    {"method": "SFT", "uses_rollouts": False, "uses_pairs": False, "required": True},
    {"method": "Rejection SFT", "uses_rollouts": True, "uses_pairs": False, "required": True},
    {"method": "Uniform OPD", "uses_rollouts": True, "uses_pairs": True, "required": "ablation"},
    {"method": "Adaptive OPD", "uses_rollouts": True, "uses_pairs": True, "required": True},
]
print(json.dumps(METHOD_TABLE, indent=2))


In [ ]:
append_cost_event(
    stage="notebook_preflight",
    notebook=Path(globals().get("__vsc_ipynb_file__", "unknown")).name,
    gpu_count=0,
    notes="preflight cell executed; update this ledger in each expensive stage",
)


## Exit gate

- Pair counts by bucket are visible.
- Rejection SFT and OPD share the same rollout cache.
- OPD reports accuracy, parseability, KL proxy, and cost.
